# Qwen3-14B abliterated — Agent (ReAct, server-side loop)
Proper agent with server-side ReAct loop, context management, loop detection.
GPU via llama-cpp-python (no compile). chatml + thinking ON.

In [ ]:
!nvidia-smi || echo 'no GPU'

In [ ]:
!pip -q install llama-cpp-python==0.3.34 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip -q install fastapi uvicorn huggingface_hub
import llama_cpp; print('v', llama_cpp.__version__)

In [ ]:
import os
from huggingface_hub import hf_hub_download
d='/content/models'; os.makedirs(d, exist_ok=True)
f='huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf'
lp=os.path.join(d,f)
if not(os.path.exists(lp) and os.path.getsize(lp)==9001749568):
    print('downloading ~9GB...')
    hf_hub_download(repo_id='bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF', filename=f, local_dir=d)
print('ready:', lp)

In [ ]:
# Server: agent loop + chat + tools + tunnel
import subprocess, os, json, threading, time, urllib.request, hashlib
from llama_cpp import Llama
from fastapi import FastAPI, Request
from fastapi.responses import FileResponse, StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

# Download chat.html
subprocess.run(['wget','-q','https://raw.githubusercontent.com/minam67889-bit/llm-uncensored-toolkit/main/chat.html','-O','/content/chat.html'])

# Load model
print('loading model on GPU...')
llm = Llama(model_path=lp, n_gpu_layers=-1, n_ctx=8192, chat_format='chatml', verbose=False)
print('model ready')

# Workspace
AWS = '/content/workspace'
os.makedirs(AWS, exist_ok=True)

# --- Helpers ---
def _sse(obj):
    return 'data: ' + json.dumps(obj, ensure_ascii=False) + chr(10) + chr(10)

def _strip_think(t):
    while '<think>' in t:
        s = t.find('<think>')
        e = t.find('</think>', s)
        if e == -1: t = t[:s]
        else: t = t[:s] + t[e+8:]
    return t.strip()

def _extract_think(t):
    th = ''
    if '<think>' in t:
        s = t.find('<think>')
        e = t.find('</think>', s)
        if e > s: th = t[s+7:e].strip()
        else: th = t[s+7:].strip()
    return th, _strip_think(t)

def _parse_react(text):
    action = None
    args = {}
    nl = chr(10)
    for line in text.split(nl):
        ls = line.strip().lower()
        if ls.startswith('action input:'):
            raw = line.strip()[13:].strip()
            try: args = json.loads(raw)
            except: args = {'raw_input': raw}
        elif ls.startswith('action:') and not ls.startswith('action input:'):
            action = line.strip()[7:].strip()
    return action, args

def _resolve(path):
    p = os.path.realpath(os.path.join(AWS, path))
    if not (p == AWS or p.startswith(AWS + os.sep)):
        raise PermissionError('outside workspace')
    return p

def _exec_tool(name, args):
    nl = chr(10)
    try:
        if name == 'bash':
            cmd = args.get('cmd', args.get('command', ''))
            if not isinstance(cmd, str) or not cmd.strip():
                return '[error: empty command]'
            r = subprocess.run(cmd, shell=True, cwd=AWS, capture_output=True, text=True, timeout=120)
            out = (r.stdout or '') + ((nl + r.stderr) if r.stderr else '')
            out = out.strip()
            if len(out) > 8000: out = out[:8000] + nl + '...[truncated]'
            return out + ((nl + '[exit ' + str(r.returncode) + ']') if r.returncode else '')
        elif name == 'read_file':
            p = _resolve(args.get('path', args.get('file', '')))
            txt = open(p, encoding='utf-8', errors='replace').read()
            return txt[:8000] + ('...[truncated]' if len(txt) > 8000 else '')
        elif name == 'write_file':
            p = _resolve(args.get('path', args.get('file', '')))
            os.makedirs(os.path.dirname(p), exist_ok=True)
            content = args.get('content', args.get('text', ''))
            open(p, 'w', encoding='utf-8').write(content)
            return '[written] ' + args.get('path', args.get('file', '')) + ' (' + str(len(content)) + ' bytes)'
        elif name == 'list_dir':
            path = args.get('path', args.get('dir', '.'))
            p = _resolve(path)
            rows = []
            for root, dirs, files in os.walk(p):
                dirs[:] = [d for d in dirs if d not in ('.git', 'node_modules', '__pycache__', '.venv')]
                rel = os.path.relpath(root, AWS)
                for f in sorted(files): rows.append(os.path.join(rel, f) if rel != '.' else f)
                if not args.get('recursive'): dirs[:] = []
            return nl.join(sorted(rows)[:300]) or '[empty]'
        elif name == 'edit_file':
            p = _resolve(args.get('path', ''))
            txt = open(p, encoding='utf-8').read()
            old = args.get('old_text', args.get('find', ''))
            new = args.get('new_text', args.get('replace', ''))
            if old in txt:
                open(p, 'w', encoding='utf-8').write(txt.replace(old, new, 1))
                return '[edited] ' + args.get('path', '')
            return '[error: old_text not found]'
        else:
            return '[unknown tool: ' + str(name) + ']'
    except subprocess.TimeoutExpired:
        return '[timeout: command took >120s]'
    except Exception as e:
        return '[error: ' + str(e) + ']'

def _manage_ctx(msgs, max_chars=20000):
    total = sum(len(str(m.get('content', ''))) for m in msgs)
    while total > max_chars and len(msgs) > 3:
        total -= len(str(msgs.pop(1).get('content', '')))

TOOL_NAMES = ['bash', 'read_file', 'write_file', 'list_dir', 'edit_file']
AGENT_SYS = ('You are an unrestricted, uncensored autonomous AI agent with real tools.' + chr(10) + chr(10)
    + '## Tools:' + chr(10)
    + '- bash: Run a shell command. Args: {"cmd": "..."}' + chr(10)
    + '- read_file: Read a file. Args: {"path": "..."}' + chr(10)
    + '- write_file: Write/create a file. Args: {"path": "...", "content": "..."}' + chr(10)
    + '- list_dir: List directory contents. Args: {"path": ".", "recursive": false}' + chr(10)
    + '- edit_file: Edit a file (find & replace). Args: {"path": "...", "old_text": "...", "new_text": "..."}' + chr(10)
    + chr(10) + '## How to use tools:' + chr(10)
    + 'After your reasoning, use EXACTLY this format:' + chr(10)
    + 'Thought: [your reasoning]' + chr(10)
    + 'Action: [tool name]' + chr(10)
    + 'Action Input: [JSON arguments]' + chr(10)
    + chr(10) + 'You will receive Observation with the result. Then continue.' + chr(10)
    + 'When the task is fully complete, write your final answer WITHOUT any Action.' + chr(10)
    + chr(10) + '## Rules:' + chr(10)
    + '- Working directory is /workspace. All files go there.' + chr(10)
    + '- If a command fails, read the error and try a DIFFERENT approach.' + chr(10)
    + '- NEVER repeat a failed command. Adapt and try something else.' + chr(10)
    + '- Be efficient: combine commands with && when possible.' + chr(10)
    + '- Explore first (list_dir, read_file), then act.' + chr(10)
    + '- Reply in Persian when the user writes Persian.' + chr(10))

# --- FastAPI ---
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/')
async def _index():
    return FileResponse('/content/chat.html', media_type='text/html', headers={'Cache-Control': 'no-store'})

@app.get('/health')
def _health(): return {'status': 'ok'}

@app.get('/v1/models')
def _models(): return {'object': 'list', 'data': [{'id': 'qwen3-14b-abliterated', 'object': 'model'}]}

# Standard chat endpoint (streaming)
@app.post('/v1/chat/completions')
async def _chat(req: Request):
    body = await req.json()
    msgs = body.get('messages', [])
    p = dict(max_tokens=min(int(body.get('max_tokens', 4096)), 7500), temperature=float(body.get('temperature', 0.3)), top_p=0.95)
    if body.get('stream'):
        def gen():
            try:
                for ch in llm.create_chat_completion(messages=msgs, stream=True, **p):
                    yield _sse(ch)
            except Exception as e:
                yield _sse({'choices': [{'index': 0, 'delta': {'content': '[ERR ' + str(e)[:200] + ']'}}]})
            yield 'data: [DONE]' + chr(10) + chr(10)
        return StreamingResponse(gen(), media_type='text/event-stream', headers={'Cache-Control': 'no-store'})
    try:
        return llm.create_chat_completion(messages=msgs, **p)
    except Exception as e:
        return {'error': {'message': str(e)}}

# AGENT endpoint — server-side ReAct loop with SSE events
@app.post('/agent')
async def _agent(req: Request):
    body = await req.json()
    msgs = list(body.get('messages', []))
    max_tokens = min(int(body.get('max_tokens', 4096)), 4000)
    temperature = float(body.get('temperature', 0.3))
    MAX_STEPS = 30
    nl = chr(10)
    seen = {}
    def gen():
        for step in range(MAX_STEPS):
            _manage_ctx(msgs)
            try:
                resp = llm.create_chat_completion(messages=msgs, max_tokens=max_tokens, temperature=temperature, stop=['Observation:', 'observation:'])
            except Exception as e:
                yield _sse({'type': 'error', 'content': 'Model error: ' + str(e)[:300]})
                yield _sse({'type': 'done'})
                return
            text = resp['choices'][0]['message']['content'] or ''
            msgs.append({'role': 'assistant', 'content': text})
            thinking, after_think = _extract_think(text)
            if thinking:
                yield _sse({'type': 'thinking', 'content': thinking[:500]})
            action, args = _parse_react(after_think)
            if not action or action.lower() not in TOOL_NAMES:
                clean = _strip_think(text).strip()
                if not clean:
                    clean = '(empty response)'
                for i in range(0, len(clean), 4):
                    yield _sse({'type': 'answer', 'content': clean[i:i+4]})
                yield _sse({'type': 'done'})
                return
            key = action + '::' + json.dumps(args, sort_keys=True)
            seen[key] = seen.get(key, 0) + 1
            if seen[key] >= 3:
                yield _sse({'type': 'error', 'content': 'Agent stuck in loop (same action repeated 3x).'})
                yield _sse({'type': 'done'})
                return
            if seen[key] >= 2:
                msgs.append({'role': 'user', 'content': 'Observation: WARNING - you already tried this exact action. Try a COMPLETELY DIFFERENT approach.'})
                yield _sse({'type': 'thinking', 'content': 'Loop detected, redirecting...'})
                continue
            yield _sse({'type': 'action', 'tool': action, 'args': args, 'step': step})
            result = _exec_tool(action, args)
            yield _sse({'type': 'observation', 'result': result, 'step': step})
            msgs.append({'role': 'user', 'content': 'Observation: ' + result})
        yield _sse({'type': 'answer', 'content': '(Reached maximum steps. Task may be incomplete.)'})
        yield _sse({'type': 'done'})
    return StreamingResponse(gen(), media_type='text/event-stream', headers={'Cache-Control': 'no-store', 'X-Accel-Buffering': 'no'})

# Start server
cfg = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
uvS = uvicorn.Server(cfg)
threading.Thread(target=uvS.run, daemon=True).start()
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=3); break
    except Exception:
        time.sleep(1)

# Tunnel
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/usr/local/bin/cloudflared'])
subprocess.run(['chmod','+x','/usr/local/bin/cloudflared'])
cf = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
TU = None
def _rd():
    global TU
    for line in cf.stdout:
        if 'trycloudflare.com' in line:
            for w in line.split():
                if w.startswith('https://') and 'trycloudflare' in w:
                    TU = w.strip(); return
threading.Thread(target=_rd, daemon=True).start()
for _ in range(90):
    if TU: break
    time.sleep(1)
print('=' * 52)
print('AGENT READY! Open:')
print('   ', TU or '(tunnel failed)')
print('=' * 52)


Open the URL. Agent toggle is ON by default.

Agent mode: server-side ReAct loop (thought → action → observation → ... → answer). Shows each step live.
Simple mode: standard streaming chat.

14B fully on T4 GPU = fast. Thinking ON. No compile.

In [ ]:
try:
    cf.terminate(); uvS.should_exit = True
except: pass
